# Stage 2: Leakage-Aware Cross-Validation and Baseline Models
**B.Tech Project in Metallurgical and Materials Engineering, IIT Roorkee**
**Project:** Machine-Learning Prediction of Mechanical Properties of Steels from Composition, with Inverse Alloy Design

---
### Key Methodological Highlights:
1. **Near-Twin Leakage Trap:** Naive random k-fold cross-validation inflates performance because near-duplicate alloys appear across train and test folds.
2. **Grouped Cross-Validation:** Single-linkage clustering groups alloys within $\tau = 0.50$ wt% into 201 clusters, forming the headline evaluation protocol.
3. **Collinearity Audit:** Dropping Iron (Fe) from linear models removes the sum-to-100 constraint singularity.
4. **Applicability Domain Audit:** Statistically auditing distance-error relationships at the cluster level.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load processed results
metrics_df = pd.read_csv(os.path.join('..', 'results', 'stage2_metrics_summary.csv'))
ad_audit_df = pd.read_csv(os.path.join('..', 'results', 'stage2_ad_cluster_audit.csv'))
paired_df = pd.read_csv(os.path.join('..', 'results', 'stage2_paired_model_comparisons.csv'))

print("Stage 2 Results Loaded Successfully.")

Stage 2 Results Loaded Successfully.


### 1. Leakage Gap: Naive vs. Repeated Grouped CV
Comparing naive repeated k-fold CV with repeated grouped CV reveals that naive CV underestimates error by 15.5 MPa RMSE for Random Forest and 17.4 MPa for XGBoost.

In [2]:
display(metrics_df[['Model', 'Feature_Set', 'CV_Scheme', 'RMSE_Mean', 'RMSE_Std', 'MAE_Mean', 'R2_Mean']])

,Model,Feature_Set,CV_Scheme,RMSE_Mean,RMSE_Std,MAE_Mean,R2_Mean
0,Dummy (Mean),wt% (All 14),Naive Repeated (5x5),300.880967,28.538313,229.996440,-0.016523
1,Ridge (with Fe),"wt% (All 14, Collinear)",Naive Repeated (5x5),240.725533,21.857841,176.172004,0.341179
2,Ridge (without Fe),"wt% (13 non-Fe, Independent)",Naive Repeated (5x5),235.105203,19.986596,168.946623,0.371249
3,Random Forest,wt% (All 14),Naive Repeated (5x5),142.803256,15.378150,101.472231,0.764273
4,XGBoost,wt% (All 14),Naive Repeated (5x5),145.163000,17.207815,103.639402,0.756619
5,Dummy (Mean),wt% (All 14),GroupKFold (Headline 5x5),300.321282,49.893634,232.001889,-0.058918
6,Ridge (with Fe),"wt% (All 14, Collinear)",GroupKFold (Headline 5x5),245.826214,45.303041,180.029783,0.278197
7,Ridge (without Fe),"wt% (13 non-Fe, Independent)",GroupKFold (Headline 5x5),239.176056,45.091520,173.014596,0.320461
8,Random Forest,wt% (All 14),GroupKFold (Headline 5x5),158.304570,26.326628,118.579433,0.684250
9,XGBoost,wt% (All 14),GroupKFold (Headline 5x5),162.585625,30.618096,122.906042,0.669115


### 2. Paired Model Comparisons Across 25 Splits
Evaluating whether Ridge (without Fe) differs from Ridge (with Fe), and whether Random Forest and XGBoost differ. These are uncorrected paired tests. Stage 3 applies the Nadeau-Bengio correction, under which the Ridge difference is no longer significant.

In [3]:
display(paired_df[['Comparison', 'Mean_Paired_Diff', 'CI_95_Low', 'CI_95_High', 'Paired_t_pvalue', 'Distinguishable_from_Zero']])

,Comparison,Mean_Paired_Diff,CI_95_Low,CI_95_High,Paired_t_pvalue,Distinguishable_from_Zero
0,Ridge: with Fe vs without Fe,6.650159,4.366324,9.069741,0.000012,YES
1,RF vs XGBoost (wt% All 14),4.281055,-0.458366,9.075713,0.101921,NO (CI includes 0)
2,RF vs XGBoost (Atomic Fractions),-1.160361,-5.667016,3.890843,0.648930,NO (CI includes 0)
3,RF vs XGBoost (wt% + Physical),-1.261923,-6.149478,3.253055,0.609563,NO (CI includes 0)


### 3. Applicability Domain Statistical Audit (Side-by-Side)
Side-by-side comparison of pooled metrics (inflated sample size) versus corrected cluster-level metrics (independent alloy averages & cluster-level bootstrap/permutation).

In [4]:
display(ad_audit_df[['Evaluation_Level', 'Metric_Definition', 'Validity', 'Pearson_r', 'Delta_MAE_MPa', 'Delta_MAE_95_CI_low', 'Delta_MAE_95_CI_high', 'Delta_MAE_p_value', 'Significant_05']])

,Evaluation_Level,Metric_Definition,Validity,Pearson_r,Delta_MAE_MPa,Delta_MAE_95_CI_low,Delta_MAE_95_CI_high,Delta_MAE_p_value,Significant_05
0,Previous Pooled (n=7800 predictions across 25 ...,1. Raw wt% with Fe (14 elements),"INVALID (Inflated sample size, pools repeats &...",0.127454,8.887406,-0.714489,24.004124,0.0516,NO
1,"Corrected Cluster-Level (n=312 alloys, 201 clu...",1. Raw wt% with Fe (14 elements),VALID (Independent alloy averages & cluster-le...,0.129726,17.724842,-17.699891,57.199610,0.1810,NO
2,Previous Pooled (n=7800 predictions across 25 ...,2. Raw wt% non-Fe (13 elements),"INVALID (Inflated sample size, pools repeats &...",0.127641,34.989336,22.366930,45.437708,0.0000,YES
3,"Corrected Cluster-Level (n=312 alloys, 201 clu...",2. Raw wt% non-Fe (13 elements),VALID (Independent alloy averages & cluster-le...,0.129240,31.349515,-3.579874,63.398334,0.0542,NO
4,Previous Pooled (n=7800 predictions across 25 ...,3. Standardized non-Fe (13 elements),"INVALID (Inflated sample size, pools repeats &...",0.144985,63.731079,53.769967,73.878134,0.0000,YES
5,"Corrected Cluster-Level (n=312 alloys, 201 clu...",3. Standardized non-Fe (13 elements),VALID (Independent alloy averages & cluster-le...,0.155091,66.110150,19.575462,98.564946,0.0000,YES
